# Data Loading

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os

DATASET_DIR = "/kaggle/input/competitions/icdar-2026-circleid-pen-classification"
train_1 = pd.read_csv("/kaggle/input/competitions/icdar-2026-circleid-pen-classification/train.csv")
train_2 = pd.read_csv("/kaggle/input/competitions/icdar-2026-circleid-pen-classification/additional_train.csv")

train_df = pd.concat([train_1, train_2], ignore_index=True)
test_df = pd.read_csv("/kaggle/input/competitions/icdar-2026-circleid-pen-classification/test.csv")

del(train_1)
del(train_2)

In [2]:
print(f"train_df shape: {train_df.shape}")
print(f"test_df shape: {test_df.shape}")

train_df.sample(7)

train_df shape: (40250, 4)
test_df shape: (5905, 2)


,image_id,image_path,writer_id,pen_id
22533,38078,images/38078.png,W46,5
20511,34643,images/34643.png,W42,4
3487,5862,images/05862.png,W33,1
22464,37966,images/37966.png,W19,2
12393,20940,images/20940.png,W06,1
35078,27558,images/27558.png,-1,6
6830,11518,images/11518.png,W06,7


In [3]:
train_df['label'] = train_df['pen_id'].astype(int)-1

In [4]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Train/validation split

In [5]:
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 3
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_splits = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
    train_split_df = train_df.iloc[train_idx].reset_index(drop=True)
    val_split_df = train_df.iloc[val_idx].reset_index(drop=True)
    fold_splits.append((train_split_df, val_split_df))
    print(f"Fold {fold}: train={train_split_df.shape}, val={val_split_df.shape}")


(34212, 5) (6038, 5)


# Dataset creation

In [6]:
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms

class CircleDataset(Dataset):

    def __init__(self, df, img_root, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.root = Path(img_root)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.root/row["image_path"]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, row["image_id"]

        label = row["label"]

        return image, label

# Image Transformation

In [7]:
# import cv2
# import numpy as np
# from PIL import Image

# class EdgeEnhance:

#     def __call__(self, img):

#         img = np.array(img)

#         blurred = cv2.GaussianBlur(img, (0,0), 1.0)

#         enhanced = cv2.addWeighted(img, 1.5, blurred, -0.5, 0)

#         return Image.fromarray(enhanced)

In [8]:
from torchvision import transforms

normalize = transforms.Normalize(
    mean=[0.485,0.456,0.406],
    std=[0.229,0.224,0.225]
)

train_tfms = transforms.Compose([

    transforms.RandomResizedCrop(
        224,
        scale=(0.95,1.0),
        ratio=(0.95,1.05)
    ),

    # EdgeEnhance(),

    transforms.RandomRotation(12),

    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.03,0.03),
        scale=(0.97,1.03)
    ),

    transforms.ColorJitter(
        brightness=0.08,
        contrast=0.08
    ),

    transforms.ToTensor(),

    normalize
])

val_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    normalize
])

# Loading Model

In [9]:
import timm
import torch.nn as nn

def build_model(model_name="resnext"):

    if model_name == "resnext":
        model = timm.create_model("resnext26ts", pretrained=True, num_classes=8)

    elif model_name == "convnext":
        from torchvision.models import convnext_tiny
        model = convnext_tiny(weights="IMAGENET1K_V1")
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, 8)

    return model

# Model Training

In [10]:
import torch
import torch.nn.functional as F
from tqdm import tqdm

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0

    for images, labels in tqdm(loader):
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = F.cross_entropy(logits, labels, label_smoothing=0.05)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss/len(loader)

In [11]:
@torch.no_grad()
def validate(model, loader, device):
    model.eval()

    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)

        preds = logits.argmax(1)

        correct+= (preds == labels).sum().item()
        total+= labels.size(0)

    return correct/total

In [12]:
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 96
EPOCHS_resnext = 5
EPOCHS_convnext = 3

models = []

for fold, (train_split_df, val_split_df) in enumerate(fold_splits):

    print(f"\n========== Fold {fold + 1}/{N_FOLDS} ==========")

    train_ds = CircleDataset(train_split_df, DATASET_DIR, train_tfms)
    val_ds = CircleDataset(val_split_df, DATASET_DIR, val_tfms)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        num_workers=2,
        pin_memory=True
    )

    for model_name in ["resnext", "convnext"]:

        print(f"Training {model_name} on fold {fold + 1}")

        model = build_model(model_name).to(device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=2e-4,
            weight_decay=6e-5
        )

        if model_name == "resnext":
            EPOCHS = EPOCHS_resnext
        else:
            EPOCHS = EPOCHS_convnext

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=EPOCHS
        )

        best_val_acc = 0.0

        for epoch in range(EPOCHS):

            train_loss = train_epoch(model, train_loader, optimizer, device)

            val_acc = validate(model, val_loader, device)

            print(f"Fold {fold + 1} | {model_name} Epoch {epoch + 1}/{EPOCHS} loss {train_loss:.4f} val_acc {val_acc:.4f}")

            if val_acc > best_val_acc:
                best_val_acc = val_acc

            scheduler.step()

        print(f"Best {model_name} fold {fold + 1} val_acc: {best_val_acc:.4f}")
        models.append(model)


Training resnext


model.safetensors:   0%|          | 0.00/41.3M [00:00<?, ?B/s]

100%|██████████| 356/356 [03:04<00:00,  1.93it/s]


resnext Epoch 0 loss 0.6150 val_acc 0.8920


100%|██████████| 356/356 [02:33<00:00,  2.32it/s]


...

convnext Epoch 4 loss 0.4083 val_acc 0.9280


100%|██████████| 356/356 [14:06<00:00,  2.38s/it]


convnext Epoch 5 loss 0.3798 val_acc 0.9349


100%|██████████| 356/356 [14:06<00:00,  2.38s/it]


convnext Epoch 6 loss 0.3624 val_acc 0.9420
Best convnext val_acc: 0.9420


In [13]:
test_ds = CircleDataset(test_df, DATASET_DIR, val_tfms, is_test=True)

test_loader = DataLoader(test_ds, batch_size=128)

import numpy as np

preds = []

for model in models:

    model.eval()

    fold_preds = []

    with torch.no_grad():

        for images, _ in test_loader:

            images = images.to(device)

            probs = []

            # original
            logits = model(images)
            probs.append(torch.softmax(logits,1))

            # horizontal flip
            logits = model(torch.flip(images, dims=[3]))
            probs.append(torch.softmax(logits,1))

            # vertical flip
            logits = model(torch.flip(images, dims=[2]))
            probs.append(torch.softmax(logits,1))

            # 90° rotation
            logits = model(torch.rot90(images, 1, [2,3]))
            probs.append(torch.softmax(logits,1))

            prob = torch.mean(torch.stack(probs), dim=0)

            fold_preds.append(prob.cpu().numpy())

    preds.append(np.concatenate(fold_preds))

preds = np.mean(preds, axis=0)

labels = preds.argmax(1) + 1


# Submission

In [14]:
submission = pd.DataFrame({
    "image_id": test_df.image_id,
    "pen_id": labels
})

submission.to_csv("submission.csv", index=False)

In [15]:
submission.head(5)

,image_id,pen_id
0,v2_8eb750cb7bac5c42036af72b8253976b,2
1,v2_04e19b0acea03fe2ae474ce8a4c6b705,8
2,v2_6b3400d5252124adcc9859cbc78c5d8a,1
3,v2_79025cb2ef36af3dc15c91056fa225dc,8
4,v2_3ed2f62c32c69ec191b7e5b86433cb87,5
